# AI Dataset Studio - Fine-tuning Qwen3

But: entrainer un adapter LoRA pour specialiser Qwen3 sur les recommandations JSON de AI Dataset Studio.

Avant de lancer les cellules: Runtime > Change runtime type > GPU.

## 1. Verifier que le GPU est actif

Si cette cellule affiche `cuda=False`, il faut changer le runtime en GPU avant de continuer.

In [ ]:
import sys
import torch

print('Python:', sys.version)
print('Torch:', torch.__version__)
print('cuda=', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('STOP: active un runtime GPU, puis relance cette cellule.')

## 2. Envoyer le paquet du projet

Clique sur le bouton de selection et choisis `ai_dataset_studio_finetune_pack.zip` depuis ton PC.

In [ ]:
from google.colab import files

uploaded = files.upload()
print('Fichiers recus:', list(uploaded))

## 3. Extraire et verifier les fichiers

In [ ]:
from pathlib import Path
import zipfile

zip_candidates = [name for name in uploaded if name.endswith('.zip')]
assert zip_candidates, 'Upload le fichier ai_dataset_studio_finetune_pack.zip'
zip_path = zip_candidates[0]

with zipfile.ZipFile(zip_path) as archive:
    archive.extractall('.')

required = [
    'ai_lab/training/qwen3_sft_train.jsonl',
    'ai_lab/training/qwen3_sft_validation.jsonl',
    'ai_lab/training/qwen3_sft_test.jsonl',
    'ai_lab/training/cases_test.jsonl',
    'ai_lab/fine_tuning/run_ms_swift_sft.sh',
    'ai_lab/system_prompt.txt',
    'ai_lab/benchmark_ollama.py',
]
for item in required:
    print(item, 'OK' if Path(item).exists() else 'MANQUANT')

## 4. Installer les outils d'entrainement

Cette cellule peut prendre plusieurs minutes.

In [ ]:
!python -m pip install -U pip
!pip install -U ms-swift transformers

## 5. Mini test rapide

On entraine d'abord sur 12 exemples seulement. Le but est de verifier que le pipeline marche avant de lancer l'entrainement complet.

In [ ]:
from pathlib import Path

src = Path('ai_lab/training/qwen3_sft_train.jsonl')
dst = Path('ai_lab/training/qwen3_sft_train_smoke.jsonl')
lines = src.read_text(encoding='utf-8').splitlines()
dst.write_text('\n'.join(lines[:12]) + '\n', encoding='utf-8')
print('Exemples smoke:', min(12, len(lines)))

In [ ]:
!DTYPE=float16 EPOCHS=1 TRAIN_DATA=ai_lab/training/qwen3_sft_train_smoke.jsonl VAL_DATA=ai_lab/training/qwen3_sft_validation.jsonl OUTPUT_DIR=ai_lab/fine_tuning/output/qwen3_ai_dataset_studio_lora_smoke EVAL_STEPS=5 SAVE_STEPS=5 GRAD_ACCUM=4 MAX_LENGTH=2048 bash ai_lab/fine_tuning/run_ms_swift_sft.sh

## 6. Entrainement complet

Lance cette cellule seulement si le mini test a fini sans erreur.

In [ ]:
!DTYPE=float16 EPOCHS=3 OUTPUT_DIR=ai_lab/fine_tuning/output/qwen3_ai_dataset_studio_lora EVAL_STEPS=20 SAVE_STEPS=20 GRAD_ACCUM=8 MAX_LENGTH=2048 bash ai_lab/fine_tuning/run_ms_swift_sft.sh

## 7. Trouver le dernier checkpoint

Copie-colle-moi la sortie de cette cellule apres l'entrainement.

In [ ]:
from pathlib import Path

checkpoints = sorted(Path('ai_lab/fine_tuning/output').glob('**/checkpoint-*'), key=lambda p: p.stat().st_mtime)
if not checkpoints:
    print('Aucun checkpoint trouve')
else:
    print('Dernier checkpoint:')
    print(checkpoints[-1])